In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_transactions AS
SELECT
  order_id,
  item_id,
  quantity,
  price,
  order_timestamp,
  CAST(order_timestamp AS DATE) AS order_date
FROM capstone.silver.transactions;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_txn_with_customer_product AS
SELECT
  t.order_id,
  t.order_date,
  t.item_id,
  p.product_name,
  p.category AS product_category,
  c.customer_key AS customer_key,
  c.customer_id AS customer_id,
  c.name AS customer_name,
  c.email AS customer_email,
  c.region AS customer_region,
  t.quantity,
  t.price,
  (t.quantity * t.price) AS line_total,
  t.order_timestamp
FROM vw_transactions t
LEFT JOIN capstone.silver.customers_scd2 c
  ON t.order_timestamp >= c.start_date
  AND (c.end_date IS NULL OR t.order_timestamp < c.end_date)
LEFT JOIN capstone.silver.products p
  ON t.item_id = p.item_id;

In [0]:
%sql
DROP TABLE IF EXISTS capstone.gold.daily_sales_fact;
CREATE TABLE capstone.gold.daily_sales_fact
USING DELTA
CLUSTER BY (order_date, product_category, customer_id)
AS
SELECT
  order_id,
  order_date,
  item_id,
  product_name,
  product_category,
  customer_key,
  customer_id,
  customer_name,
  customer_email,
  customer_region AS region,
  quantity,
  price,
  line_total,
  order_timestamp
FROM vw_txn_with_customer_product;

In [0]:
%sql
SELECT * FROM capstone.gold.daily_sales_fact;

In [0]:
print("Gold Table Counts:")
print("capstone.gold.daily_sales_fact: ", spark.table("capstone.gold.daily_sales_fact").count())